# Customer Churn Prediction - Complete ML Pipeline

## Project Overview
Goal: Predict which customers will leave (churn) a telecom company
Model: Logistic Regression
Output: Yes/No with probability score

## What is Churn?
Churn = customers who stop using the service. Companies want to identify these customers BEFORE they leave so they can offer discounts or improve service.

## What This Notebook Covers
- Load and explore customer data
- Handle categorical data with One-Hot Encoding
- Train Logistic Regression model
- Evaluate model performance
- Predict for multiple new customers
- Save model for future use

---

## Step 1: Import Libraries

Why: We need these tools for data handling, visualization, modeling, and evaluation.

In [ ]:
# Import all required libraries
import pandas as pd                    # For data manipulation (like Excel in Python)
from matplotlib import pyplot as plt   # For creating graphs and visualizations
from sklearn.model_selection import train_test_split  # To split data into train/test
from sklearn.preprocessing import StandardScaler      # To scale features (make them comparable)
from sklearn.linear_model import LogisticRegression   # The ML model we'll use
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix  # To check model quality

print("All libraries imported successfully")

## Step 2: Load Data

Why: We need to read the CSV file into a DataFrame (like opening Excel in Python)
What to check: First few rows, column names, data types

In [ ]:
# Load customer data from CSV file
df = pd.read_csv("Customer_churn.csv")

print("="*60)
print("ORIGINAL DATA - First 5 Rows")
print("="*60)
print(df.head())
print("\nColumn Names:", df.columns.tolist())
print("Dataset Shape:", df.shape, "rows x columns")

## Step 3: Exploratory Data Analysis (EDA)

Why: Understand our data before building models
What we check: Statistics, missing values, data types

In [ ]:
print("="*60)
print("BASIC STATISTICS")
print("="*60)
print(df.describe())
# describe() shows: count, mean, std, min, 25%, 50%, 75%, max for numerical columns

print("\n" + "="*60)
print("MISSING VALUES CHECK")
print("="*60)
print(df.isnull().sum())
# isnull().sum() counts how many empty values in each column
# 0 = no missing values (good)

print("\n" + "="*60)
print("DATA TYPES INFO")
print("="*60)
print(df.info())
# info() shows column names, data types, and non-null counts

## Step 4: One-Hot Encoding

Why: Machine Learning models only understand numbers, not text like 'Month-to-month' or 'Yes'
How: Convert each category into a separate column with 0/1 values

Example: 'contract_type' column:
- Month-to-month -> [1, 0, 0]
- One year -> [0, 1, 0]
- Two year -> [0, 0, 1]

In [ ]:
# Convert categorical columns to numbers using One-Hot Encoding

# 1. Encode contract type (Month-to-month, One year, Two year)
contract_encoded = pd.get_dummies(df["contract_type"], prefix="contract")
# prefix="contract" creates columns like: contract_Month-to-month, contract_One year, etc.

# 2. Encode paperless billing (Yes/No)
paperless_encoded = pd.get_dummies(df["paperless_billing"], prefix="billing")
# Creates: billing_Yes, billing_No

# 3. Encode payment method (Electronic check, Credit card, Bank transfer, etc.)
payment_encoded = pd.get_dummies(df["payment_method"], prefix="payment")
# Creates multiple payment_* columns

# Remove original text columns (can't use them directly)
df_encoded = df.drop(["customer_id", "contract_type", "paperless_billing", "payment_method"], axis=1)
# axis=1 means drop columns (axis=0 would drop rows)

# Combine everything: original numbers + new encoded columns
df_final = pd.concat([df_encoded, contract_encoded, paperless_encoded, payment_encoded], axis=1)
# concat joins DataFrames side by side

print("="*60)
print("FINAL DATA AFTER ENCODING")
print("="*60)
print(df_final.head())
print("\nTotal columns now:", len(df_final.columns))

## Step 5: Feature and Target Selection

Why: Separate what the model learns from (X) and what it predicts (Y)
X (Features): All columns except 'churn'
Y (Target): 'churn' column (what we want to predict)

In [ ]:
# Separate features (X) and target (Y)
X = df_final.drop("churn", axis=1)   # Features: all columns except churn
Y = df["churn"]                       # Target: churn column from original df

# Convert target from text to numbers (Yes=1, No=0)
Y = Y.map({'Yes': 1, 'No': 0})
# map() replaces values: 'Yes' becomes 1, 'No' becomes 0

# Store column names for later (when predicting new customers)
training_columns = X.columns.tolist()

print("="*60)
print("FEATURES (X) - First 5 rows")
print("="*60)
print(X.head())
print("\nFeatures shape:", X.shape)
print("\nTarget (Y) - First 10 values:")
print(Y.head(10))
print("\nChurn distribution:")
print(Y.value_counts())
print("   0 = No churn, 1 = Churn")

## Step 6: Train-Test Split

Why: We need to test on data the model hasn't seen
How: 80% for training, 20% for testing
random_state: Ensures same split every time (reproducibility)

In [ ]:
# Split data: 80% train, 20% test
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
# test_size=0.2 means 20% for testing
# random_state=42 ensures same split every time (like a fixed seed)

print("="*60)
print("TRAIN-TEST SPLIT RESULTS")
print("="*60)
print(f"Training set: {len(X_train)} samples ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test set: {len(X_test)} samples ({len(X_test)/len(X)*100:.0f}%)")

## Step 7: Feature Scaling

Why: Features have different scales (tenure: 1-70, monthly_charges: 20-120)
Without scaling: Larger numbers dominate the model
StandardScaler: Makes all features have mean=0, std=1

In [ ]:
# Create scaler object
scaler = StandardScaler()

# Fit on training data ONLY (learn mean and std from training)
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using same scaler (no new fitting!)
X_test_scaled = scaler.transform(X_test)

print("="*60)
print("FEATURE SCALING COMPLETE")
print("="*60)
print(f"Before scaling - Training data range: {X_train.min().min():.2f} to {X_train.max().max():.2f}")
print(f"After scaling - Training data range: {X_train_scaled.min():.2f} to {X_train_scaled.max():.2f}")

## Step 8: Model Training

Why: Logistic Regression is best for binary classification (Yes/No)
max_iter: Maximum iterations for convergence
random_state: Ensures reproducible results

In [ ]:
# Create and train Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42)
# max_iter=1000: enough iterations for model to learn
# random_state=42: same results each run

# Train the model
model.fit(X_train_scaled, Y_train)

print("="*60)
print("MODEL TRAINING COMPLETE")
print("="*60)
print(f"Model coefficients: {model.coef_[0][:3]}... (showing first 3)")
print(f"Model intercept: {model.intercept_[0]:.4f}")

## Step 9: Make Predictions

Why: See how well our model performs on unseen data
predict: Gives class (0 or 1)
predict_proba: Gives probability scores

In [ ]:
# Make predictions on test data
y_pred = model.predict(X_test_scaled)      # Class predictions (0 or 1)
y_prob = model.predict_proba(X_test_scaled) # Probability scores [P(No), P(Yes)]

print("="*60)
print("PREDICTIONS MADE")
print("="*60)
print("First 10 predictions vs actual:")
for i in range(10):
    pred_class = "Churn" if y_pred[i] == 1 else "Stay"
    actual_class = "Churn" if Y_test.iloc[i] == 1 else "Stay"
    prob = y_prob[i][1] * 100
    print(f"  Customer {i+1}: Predicted={pred_class} ({prob:.1f}%), Actual={actual_class}")

## Step 10: Model Evaluation

Why: Check how good our model is

Metrics Explained:
- Accuracy: Percentage of correct predictions
- Precision: When model says Churn, how often it is correct
- Recall: How many actual churners did we catch
- F1-Score: Balance of precision and recall
- Confusion Matrix: Shows correct/wrong predictions in a table

In [ ]:
print("="*60)
print("MODEL EVALUATION RESULTS")
print("="*60)

# Accuracy score
acc = accuracy_score(Y_test, y_pred)
print(f"Accuracy: {acc:.4f} ({acc*100:.1f}%)")

# Classification report (precision, recall, f1-score)
print("\nClassification Report:")
print(classification_report(Y_test, y_pred, target_names=['Stay', 'Churn']))

# Confusion Matrix
cm = confusion_matrix(Y_test, y_pred)
print("Confusion Matrix:")
print("                 Predicted")
print("              Stay    Churn")
print(f"Actual Stay   {cm[0][0]:5d}   {cm[0][1]:5d}")
print(f"       Churn   {cm[1][0]:5d}   {cm[1][1]:5d}")

# Interpretation
print("\nInterpretation:")
print(f"  - Correctly predicted {cm[0][0]} customers who stayed")
print(f"  - Correctly predicted {cm[1][1]} customers who churned")
print(f"  - Missed {cm[1][0]} churners (false negatives)")
print(f"  - Wrongly flagged {cm[0][1]} customers as churn (false positives)")

## Step 11: Visualize Confusion Matrix

Why: Visual representation helps understand model performance

In [ ]:
# Create heatmap for confusion matrix
plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar()
plt.title('Confusion Matrix - Customer Churn Prediction', fontsize=14)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)

# Add text annotations
tick_marks = [0, 1]
plt.xticks(tick_marks, ['Stay', 'Churn'])
plt.yticks(tick_marks, ['Stay', 'Churn'])

for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=20, color='white' if cm[i, j] > cm.max()/2 else 'black')

plt.tight_layout()
plt.show()

## Step 12: Shape Verification

Why: Ensure all arrays have correct dimensions before evaluation

In [ ]:
print("="*60)
print("DATA SHAPES VERIFICATION")
print("="*60)
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape:  {X_test_scaled.shape}")
print(f"Y_train shape:        {Y_train.shape}")
print(f"Y_test shape:         {Y_test.shape}")
print(f"y_pred shape:         {y_pred.shape}")
print("\nAll shapes match! Test set predictions align with actual values.")

## Step 13: Predict for New Customers

Why: Real-world usage - predict churn for new customers
What we do: Encode new data same way, scale, then predict

In [ ]:
# Create 3 sample customers to predict
new_customers = pd.DataFrame([
    {
        'tenure': 8,
        'monthly_charges': 95.5,
        'total_charges': 764.0,
        'avg_monthly_gb_download': 32.5,
        'avg_calls_per_month': 58,
        'customer_service_calls': 3,
        'contract_type': 'Month-to-month',
        'paperless_billing': 'Yes',
        'payment_method': 'Electronic check'
    },
    {
        'tenure': 36,
        'monthly_charges': 65.2,
        'total_charges': 2347.2,
        'avg_monthly_gb_download': 15.2,
        'avg_calls_per_month': 38,
        'customer_service_calls': 1,
        'contract_type': 'One year',
        'paperless_billing': 'No',
        'payment_method': 'Credit card'
    },
    {
        'tenure': 60,
        'monthly_charges': 42.0,
        'total_charges': 2520.0,
        'avg_monthly_gb_download': 7.2,
        'avg_calls_per_month': 25,
        'customer_service_calls': 0,
        'contract_type': 'Two year',
        'paperless_billing': 'No',
        'payment_method': 'Bank transfer'
    }
])

print("="*60)
print("NEW CUSTOMERS TO PREDICT")
print("="*60)
print(new_customers[['tenure', 'monthly_charges', 'contract_type']])

## Step 14: Function to Encode New Customers

Why: New data must be processed exactly like training data
Important: Must have same columns in same order

In [ ]:
def encode_new_customer(df):
    """
    Convert new customer data to same format as training data
    """
    # Step 1: One-hot encode categorical columns
    contract_encoded = pd.get_dummies(df["contract_type"], prefix="contract")
    paperless_encoded = pd.get_dummies(df["paperless_billing"], prefix="billing")
    payment_encoded = pd.get_dummies(df["payment_method"], prefix="payment")
    
    # Step 2: Drop original text columns
    df_encoded = df.drop(["contract_type", "paperless_billing", "payment_method"], axis=1)
    
    # Step 3: Combine with encoded columns
    df_final = pd.concat([df_encoded, contract_encoded, paperless_encoded, payment_encoded], axis=1)
    
    # Step 4: Add missing columns (if any) with 0
    for col in training_columns:
        if col not in df_final.columns:
            df_final[col] = 0
    
    # Step 5: Return only training columns in correct order
    return df_final[training_columns]

print("Encoding function created")

## Step 15: Make Predictions for New Customers

Why: Apply the same preprocessing pipeline to new data

In [ ]:
# Encode new customers
new_encoded = encode_new_customer(new_customers)

# Scale using same scaler (no new fitting!)
new_scaled = scaler.transform(new_encoded)

# Predict
predictions = model.predict(new_scaled)
probabilities = model.predict_proba(new_scaled)

# Create results DataFrame
results = new_customers.copy()
results["churn_prediction"] = ["Yes" if p == 1 else "No" for p in predictions]
results["churn_probability"] = [f"{prob[1]*100:.1f}%" for prob in probabilities]

print("="*60)
print("PREDICTION RESULTS FOR NEW CUSTOMERS")
print("="*60)
print(results[["tenure", "monthly_charges", "contract_type", "churn_prediction", "churn_probability"]])

print("\nBusiness Interpretation:")
for i, row in results.iterrows():
    if row['churn_prediction'] == 'Yes':
        print(f"  Customer {i+1}: HIGH RISK ({row['churn_probability']}) - Offer discount")
    else:
        print(f"  Customer {i+1}: LOW RISK ({row['churn_probability']}) - Keep service as is")

## Step 16: Save Model for Future Use

Why: Don't want to retrain every time
What we save: Model, scaler, column names
Format: joblib (special format for scikit-learn models)

In [ ]:
import joblib

# Save model and preprocessing objects
joblib.dump(model, "churn_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(training_columns, "training_columns.pkl")

print("="*60)
print("MODEL SAVED SUCCESSFULLY")
print("="*60)
print("Files created:")
print("  - churn_model.pkl (the trained model)")
print("  - scaler.pkl (for scaling new data)")
print("  - training_columns.pkl (column names for new data)")

## Step 17: Load and Test Saved Model

Why: Verify we can reuse the saved model later

In [ ]:
# Load saved objects
loaded_model = joblib.load('churn_model.pkl')
loaded_scaler = joblib.load('scaler.pkl')
loaded_columns = joblib.load('training_columns.pkl')

# Test prediction with loaded model
test_customer = pd.DataFrame([{
    'tenure': 12,
    'monthly_charges': 80.5,
    'total_charges': 966.0,
    'avg_monthly_gb_download': 25.0,
    'avg_calls_per_month': 45,
    'customer_service_calls': 2,
    'contract_type': 'Month-to-month',
    'paperless_billing': 'Yes',
    'payment_method': 'Electronic check'
}])

# Encode using saved columns
test_encoded = encode_new_customer(test_customer)
test_scaled = loaded_scaler.transform(test_encoded)
test_pred = loaded_model.predict(test_scaled)[0]
test_prob = loaded_model.predict_proba(test_scaled)[0][1]

print("="*60)
print("MODEL LOADED AND TESTED")
print("="*60)
print(f"Test customer: 12 months, Month-to-month contract")
print(f"Prediction: {'CHURN' if test_pred == 1 else 'STAY'}")
print(f"Probability: {test_prob*100:.1f}% chance of churn")

## PROJECT SUMMARY

### What We Accomplished:

| Step | Task | Status |
|------|------|--------|
| 1 | Load and explore data | Completed |
| 2 | Handle missing values | Completed |
| 3 | One-hot encode categories | Completed |
| 4 | Train-test split | Completed |
| 5 | Feature scaling | Completed |
| 6 | Logistic Regression training | Completed |
| 7 | Model evaluation | Completed |
| 8 | Predict new customers | Completed |
| 9 | Save model for reuse | Completed |

### Key Takeaways:

1. Logistic Regression works well for binary classification (Yes/No)
2. One-hot encoding is essential for categorical data
3. Feature scaling ensures all features contribute equally
4. Confusion matrix helps understand where model makes mistakes
5. Model saving allows reuse without retraining

### Business Impact:

Companies can use this model to:
- Identify high-risk customers BEFORE they leave
- Offer targeted discounts to at-risk customers
- Reduce customer churn and increase revenue
- Save marketing costs by focusing on high-risk customers

---

Project Complete. Model ready for deployment.